# PyTorch 基础 + ViT 视觉特征提取

**学习目标**：用预训练 ViT 提取网页截图的视觉特征，这是论文视觉编码器模块的核心操作。

完成本 notebook 后你将掌握：
- PyTorch Tensor 的基本操作
- 如何加载和处理图像
- 如何使用 HuggingFace 加载预训练模型
- 如何提取图像特征向量（用于后续的跨模态对齐）

## Part 1：PyTorch Tensor 基础

Tensor 是 PyTorch 的核心数据结构，类似 numpy 数组，但可以在 GPU 上运算。
在论文中，图像、特征向量、注意力权重全都是 Tensor。

In [ ]:
import torch

# 查看运行设备（Mac 上用 MPS 加速，Windows 4080 上用 CUDA）
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'使用设备: {device}')
print(f'PyTorch 版本: {torch.__version__}')

In [ ]:
# Tensor 基本操作
# 一张图像在 PyTorch 中的表示是 [C, H, W] 的 Tensor
# C=通道数(RGB=3)，H=高度，W=宽度

# 创建一个模拟图像 Tensor（3通道，224x224）
fake_image = torch.randn(3, 224, 224)
print(f'图像 Tensor 形状: {fake_image.shape}')   # torch.Size([3, 224, 224])
print(f'数据类型: {fake_image.dtype}')            # torch.float32
print(f'最小值: {fake_image.min():.3f}, 最大值: {fake_image.max():.3f}')

In [ ]:
# 批次（Batch）的概念：模型通常一次处理多张图像
# 批次 Tensor 的形状是 [B, C, H, W]，B 是批次大小

batch_size = 4
batch_images = torch.randn(batch_size, 3, 224, 224)
print(f'批次图像形状: {batch_images.shape}')  # torch.Size([4, 3, 224, 224])

# 取第一张图像
first_image = batch_images[0]  # 形状: [3, 224, 224]
print(f'单张图像形状: {first_image.shape}')

# 增加批次维度（把单张图像变成批次=1）
single_batch = first_image.unsqueeze(0)  # 形状: [1, 3, 224, 224]
print(f'添加批次维度后: {single_batch.shape}')

In [ ]:
# 把 Tensor 移动到加速设备上
tensor_on_device = fake_image.to(device)
print(f'设备上的 Tensor: {tensor_on_device.device}')

# 从设备移回 CPU（用于打印、存储、numpy 转换）
tensor_on_cpu = tensor_on_device.cpu()
print(f'移回 CPU: {tensor_on_cpu.device}')

## Part 2：加载和处理真实网页截图

在论文的数据管线中，输入是真实网页的截图。
这里我们用 Pillow 加载图像，再用 HuggingFace 的处理器将其转换为模型需要的格式。

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
import os

# 用一张示例网页截图来演示
# 实际项目中，这里会是从 WebCode2M 加载的网页截图
sample_image_path = '../data/raw/sample_webpage.png'

# 如果没有样本图像，创建一个模拟的网页截图（白底+色块，像素化网页布局）
if not os.path.exists(sample_image_path):
    os.makedirs('../data/raw', exist_ok=True)
    # 创建一个模拟网页截图（1280x800，白色背景）
    import numpy as np
    mock_webpage = np.ones((800, 1280, 3), dtype=np.uint8) * 255  # 白色背景
    # 添加导航栏（深色）
    mock_webpage[0:60, :] = [30, 30, 30]
    # 添加主图区域（蓝色）
    mock_webpage[80:400, 50:800] = [70, 130, 180]
    # 添加文字区域（浅灰）
    mock_webpage[80:400, 850:1230] = [240, 240, 240]
    # 添加三个卡片（绿色）
    for i in range(3):
        x = 50 + i * 420
        mock_webpage[450:700, x:x+380] = [100, 180, 100]
    image = Image.fromarray(mock_webpage)
    image.save(sample_image_path)
    print(f'已创建模拟网页截图: {sample_image_path}')
else:
    image = Image.open(sample_image_path)
    print(f'已加载图像: {sample_image_path}')

print(f'图像尺寸: {image.size}')   # (宽, 高)
print(f'图像模式: {image.mode}')   # RGB

plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title('模拟网页截图')
plt.axis('off')
plt.show()

## Part 3：加载预训练 ViT 模型

ViT（Vision Transformer）把图像切分成 patch，用 Transformer 处理，输出每个 patch 的特征向量。

论文中用 ViT 作为视觉编码器，提取网页截图的视觉特征序列，供后续跨模态注意力使用。

这里使用 `google/vit-base-patch16-224`：
- base：中等大小，在 Mac/4080 上都能跑
- patch16：每个 patch 是 16x16 像素
- 224：输入图像大小 224x224

In [ ]:
from transformers import ViTModel, ViTImageProcessor

model_name = 'google/vit-base-patch16-224'

# 加载图像处理器（负责将图像缩放、归一化为模型需要的格式）
processor = ViTImageProcessor.from_pretrained(model_name)
print('图像处理器加载成功')
print(f'  期望输入尺寸: {processor.size}')
print(f'  归一化均值: {processor.image_mean}')
print(f'  归一化标准差: {processor.image_std}')

In [ ]:
# 加载 ViT 模型（只要编码器部分，不要分类头）
model = ViTModel.from_pretrained(model_name)
model = model.to(device)
model.eval()  # 推理模式，关闭 dropout

# 查看模型参数量
total_params = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {total_params / 1e6:.1f}M')
print(f'模型已加载到: {device}')

## Part 4：提取视觉特征

ViT 会把 224x224 的图像切成 14x14=196 个 patch，加上 1 个 [CLS] token，共 197 个 token。
每个 token 的特征维度是 768。

所以输出特征形状是 **[1, 197, 768]**：
- 1 = batch size
- 197 = 196个patch + 1个CLS
- 768 = 特征维度

在论文的跨模态对齐中，这 196 个 patch 特征就代表图像不同区域的视觉信息。

In [ ]:
# 用处理器将图像转换为模型输入
inputs = processor(images=image, return_tensors='pt')
print(f'处理后的输入形状: {inputs["pixel_values"].shape}')  # [1, 3, 224, 224]

# 移动到设备
inputs = {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# 前向推理，提取特征
with torch.no_grad():  # 推理时不需要计算梯度，节省内存
    outputs = model(**inputs)

# last_hidden_state: 所有 token 的特征序列
features = outputs.last_hidden_state
print(f'特征序列形状: {features.shape}')  # [1, 197, 768]

# CLS token（第0个）代表整张图像的全局特征
cls_feature = features[:, 0, :]  
print(f'全局特征（CLS）形状: {cls_feature.shape}')  # [1, 768]

# Patch features（第1-196个）代表各局部区域的特征
patch_features = features[:, 1:, :]  
print(f'局部 patch 特征形状: {patch_features.shape}')  # [1, 196, 768]

## Part 5：可视化 patch 特征

把 196 个 patch 特征还原成 14x14 的空间布局，可以看到 ViT 在图像哪些区域"关注"了什么。
这个可视化在论文实验部分会很有用。

In [ ]:
import numpy as np

# 把 patch 特征从设备移回 CPU 并转为 numpy
patch_features_np = patch_features.squeeze(0).cpu().numpy()  # [196, 768]

# 用每个 patch 特征的 L2 范数作为"激活强度"
patch_norms = np.linalg.norm(patch_features_np, axis=1)  # [196]

# 还原成 14x14 空间布局
patch_map = patch_norms.reshape(14, 14)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)
axes[0].set_title('原始网页截图')
axes[0].axis('off')

im = axes[1].imshow(patch_map, cmap='hot', interpolation='nearest')
axes[1].set_title('ViT Patch 特征激活强度（14x14）')
axes[1].set_xlabel('水平 patch 索引')
axes[1].set_ylabel('垂直 patch 索引')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print(f'激活强度范围: {patch_norms.min():.2f} ~ {patch_norms.max():.2f}')

## Part 6：批量处理多张图像

实际训练时需要批量处理，这里演示如何一次处理多张网页截图。

In [ ]:
# 模拟 3 张不同的网页截图（实际中来自 DataLoader）
images = [image, image, image]  # 暂时用同一张图演示

# 批量处理
batch_inputs = processor(images=images, return_tensors='pt')
batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
print(f'批量输入形状: {batch_inputs["pixel_values"].shape}')  # [3, 3, 224, 224]

with torch.no_grad():
    batch_outputs = model(**batch_inputs)

batch_features = batch_outputs.last_hidden_state
print(f'批量特征形状: {batch_features.shape}')  # [3, 197, 768]
print()
print('总结：')
print(f'  输入: {len(images)} 张网页截图')
print(f'  输出: {batch_features.shape[0]} x {batch_features.shape[1]} x {batch_features.shape[2]}')
print(f'       (batch) x (197个token) x (768维特征)')

## 小结

你刚刚完成了论文视觉编码器的核心操作：

| 步骤 | 操作 | 对应论文内容 |
|------|------|--------------|
| 加载图像 | `PIL.Image` | 网页截图输入 |
| 图像预处理 | `ViTImageProcessor` | 标准化为模型输入 |
| 特征提取 | `ViTModel.forward()` | 视觉编码器 |
| 特征输出 | `[B, 197, 768]` | 视觉特征序列（供跨模态注意力使用） |

**下一步（notebook 02）**：用 CodeBERT 提取 HTML 源码的特征序列，输出格式将是 `[B, N, 768]`，N 是 DOM 节点数。然后我们就可以开始实现两个序列之间的交叉注意力对齐了。